In [8]:
import pandas as pd

# Load test set
test_df = pd.read_csv("../Dataset/QK-video_test.csv")

# Ground truth: mỗi user đã click những item nào
actual_items = test_df.groupby("user_id")["item_id"].apply(list).to_dict()

# Danh sách tất cả item có thể gợi ý
all_items = sorted(test_df["item_id"].unique())

print(f"✅ Test set loaded: {len(actual_items)} users, {len(all_items)} unique items")

✅ Test set loaded: 204297 users, 98322 unique items


In [9]:
import torch
import sys
import pickle
from deepfm import DeepFM  # Đảm bảo file deepfm.py trong path
sys.path.append("../src-backend/models")

# Load feature_index và scaler
with open("../Model/Deep FM/video/feature_index.pkl", "rb") as f:
    feature_index = pickle.load(f)

with open("../Model/Deep FM/video/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

# Khởi tạo model
cat_dims = [len(feature_index[col]) for col in ['user_id', 'item_id', 'video_category', 'gender', 'age']]
num_dim = 1  # chỉ có watching_times_scaled

model = DeepFM(cat_dims=cat_dims, num_dim=num_dim)
model.load_state_dict(torch.load("../Model/Deep FM/video/deepfm_model.pth", map_location="cpu"))
model.eval()

print("✅ Loaded DeepFM model")


✅ Loaded DeepFM model


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_9092\1698075678.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("../Model/Deep FM/video/dee

In [12]:
import torch

# Chọn device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚙️ Using device: {device}")

def recommend_from_model_batch(user_row, all_items, top_k=5):
    user_id = user_row["user_id"]
    gender = user_row["gender"]
    age = user_row["age"]
    video_category = user_row["video_category"]
    watch_time = user_row["watching_times_scaled"]

    n_items = len(all_items)

    # Batch categorical tensor: [N, 5]
    cat_tensor = torch.tensor([
        [user_id, item_id, video_category, gender, age] for item_id in all_items
    ], dtype=torch.long, device=device)

    # Batch numeric tensor: [N, 1]
    num_tensor = torch.tensor([[watch_time]] * n_items, dtype=torch.float32, device=device)

    with torch.no_grad():
        scores = model(cat_tensor, num_tensor).squeeze().cpu().numpy()

    # Lấy top-K item
    top_indices = scores.argsort()[::-1][:top_k]
    return [all_items[i] for i in top_indices]


⚙️ Using device: cuda


In [ ]:
from tqdm import tqdm

# Đưa mô hình lên GPU nếu có
model.to(device)
model.eval()

recommended_items = {}
sample_user_ids = list(actual_items.keys())[:3000]  # Giới hạn 3,000 user

for user_id in tqdm(sample_user_ids):
    row = test_df[test_df["user_id"] == user_id].iloc[0]
    recs = recommend_from_model_batch(row, all_items, top_k=5)
    recommended_items[user_id] = recs


  0%|          | 6/3000 [05:11<43:12:34, 51.96s/it]


KeyboardInterrupt: 

In [ ]:
import math

def hit_rate_at_k(recommended_items, actual_items, k=5):
    hits = 0
    for user, actual in actual_items.items():
        recs = recommended_items.get(user, [])[:k]
        if any(item in actual for item in recs):
            hits += 1
    return hits / len(recommended_items)

def precision_at_k(recommended_items, actual_items, k=5):
    total = 0
    for user, actual in actual_items.items():
        recs = recommended_items.get(user, [])[:k]
        total += len([item for item in recs if item in actual]) / k
    return total / len(recommended_items)

def recall_at_k(recommended_items, actual_items, k=5):
    total = 0
    for user, actual in actual_items.items():
        recs = recommended_items.get(user, [])[:k]
        total += len([item for item in recs if item in actual]) / len(actual)
    return total / len(recommended_items)

def ndcg_at_k(recommended_items, actual_items, k=5):
    def dcg(recs, actual):
        return sum(1 / math.log2(i + 2) if rec in actual else 0 for i, rec in enumerate(recs[:k]))
    
    total = 0
    for user, actual in actual_items.items():
        ideal_dcg = sum(1 / math.log2(i + 2) for i in range(min(len(actual), k)))
        if ideal_dcg == 0:
            continue
        recs = recommended_items.get(user, [])[:k]
        total += dcg(recs, actual) / ideal_dcg
    return total / len(recommended_items)


In [ ]:
K = 5
print(f"🎯 HitRate@{K}:    {hit_rate_at_k(recommended_items, actual_items, K):.4f}")
print(f"🎯 Precision@{K}: {precision_at_k(recommended_items, actual_items, K):.4f}")
print(f"🎯 Recall@{K}:    {recall_at_k(recommended_items, actual_items, K):.4f}")
print(f"🎯 NDCG@{K}:      {ndcg_at_k(recommended_items, actual_items, K):.4f}")
